# PLN Sesión 2: Candidatos y Significados (Terminología y Vectores)

---

## ¿Por qué?

En la sesión anterior se analizó cómo el modelo identifica entidades individuales (quién es quién). Sin embargo, el lenguaje especializado rara vez se manifiesta en palabras sueltas. Conceptos como *inteligencia artificial*, *red social* o *aprendizaje automático* son unidades de significado compuestas por múltiples elementos.

Además, persistía un desafío: ¿cómo saber que dos términos distintos, como *bulo* y *fake news*, se refieren a lo mismo si no comparten ninguna característica gráfica? Esta sesión aborda la transición de la forma al concepto mediante la extracción de terminología compleja y el análisis de similitud semántica.

## ¿Qué?

Se trabajarán dos conceptos fundamentales:
1. **Extracción terminológica basada en patrones:** El uso de categorías gramaticales (POS) para identificar combinaciones frecuentes que constituyen términos técnicos.
2. **Vectores de palabras (Word Embeddings):** La representación matemática de las palabras en un espacio multidimensional donde la cercanía geométrica equivale a la cercanía de significado.

## ¿Para qué?

Para un profesional de la lengua, estas técnicas permiten:
- **Crear glosarios automáticos:** Identificar candidatos a términos técnicos en grandes volúmenes de texto especializado.
- **Agrupar por significado:** Encontrar sinónimos o conceptos relacionados sin necesidad de reglas de búsqueda exactas.
- **Analizar campos semánticos:** Entender qué conceptos orbitan alrededor de un término clave (ej. qué palabras se asocian con *tecnología* en un corpus determinado).

## ¿Cómo?

Se empleará **spaCy** para filtrar patrones morfosintácticos y para explorar la distancia vectorial entre términos, superando definitivamente la "dictadura de la forma" impuesta por los motores de búsqueda tradicionales.

---
## 1. Preparación del entorno

In [ ]:
!python -m spacy download es_core_news_sm --quiet
import spacy
import pandas as pd
from collections import Counter

nlp = spacy.load("es_core_news_sm")
print("Modelo de lengua española cargado.")

---
## 2. Extracción de terminología compleja

La mayoría de los términos técnicos no son monolemas (una palabra), sino polilemas (varias palabras). La estructura más común en español es **Sustantivo + Adjetivo** o **Sustantivo + Preposición + Sustantivo**.

In [ ]:
texto_tecnico = """
La inteligencia artificial y el aprendizaje automático están transformando la lingüística computacional.
El procesamiento de lenguaje natural requiere una infraestructura digital robusta y modelos de lengua 
entrenados con corpus lingüísticos de alta calidad. La brecha tecnológica se reduce con innovación abierta.
"""

doc = nlp(texto_tecnico)

# Extracción basada en patrones gramaticales (Sustantivo + Adjetivo)
candidatos = []
for i in range(len(doc) - 1):
    token = doc[i]
    siguiente = doc[i+1]
    if token.pos_ == "NOUN" and siguiente.pos_ == "ADJ":
        candidatos.append(f"{token.text} {siguiente.text}")

print("Candidatos a términos (Sustantivo + Adjetivo):")
print(candidatos)

---
## 2. El fin de la forma: Similitud Semántica

### ¿Cómo ve la máquina el significado? (El Mapa de Palabras)

Para que una máquina "entienda" que *bulo* y *fake news* se parecen, no analiza sus letras. Lo que hace es convertir cada palabra en una **coordenada en un mapa gigante** (un espacio vectorial).

Considérese un mapa de solo dos ejes:
- **Eje X (Tecnología):** Cuanto más a la derecha, más tecnológico.
- **Eje Y (Veracidad):** Cuanto más arriba, más verdadero.

En este mapa mental:
1. **"Noticia"** estaría arriba (verdadero) y a la derecha (tecnológico si es digital).
2. **"Bulo"** estaría abajo (falso) y a la izquierda (general).
3. **"Fake news"** estaría abajo (falso) y a la derecha (tecnológico).

Al calcular la distancia entre los puntos, la máquina descubre que **"bulo"** y **"fake news"** están mucho más cerca entre sí que de la palabra **"televisor"**. 

**La magia no es tal:** es simplemente geometría. El modelo ha aprendido estas coordenadas leyendo millones de frases y observando qué palabras suelen aparecer en los mismos "barrios" del mapa.

**Nota:** Para este análisis se requiere un modelo con vectores cargados (`es_core_news_lg`), pero podemos simular la lógica con el modelo estándar para entender el concepto.

In [ ]:
# Comparación de similitud semántica
palabras = ["bulo", "noticia", "mentira", "periódico", "televisión"]

base = nlp("fake news")
comparaciones = []

for p in palabras:
    token_p = nlp(p)
    comparaciones.append({
        "Término": p,
        "Similitud con 'fake news'": base.similarity(token_p)
    })

df_similitud = pd.DataFrame(comparaciones).sort_values(by="Similitud con 'fake news'", ascending=False)
print(df_similitud)

### Reflexión profesional
- El sistema no busca letras comunes (como haríamos con Regex).
- El sistema busca **cercanía de uso**. 
- ¿Qué utilidad tiene esto para un traductor que busca el equivalente más natural en un contexto técnico?

---
## 3. Reto: El radar semántico

La tarea consiste en procesar un texto y encontrar qué términos orbitan alrededor de un concepto clave, sin que ese concepto sea mencionado explícitamente en todas las frases.

In [ ]:
concepto_clave = "tecnología"
texto_analizar = """
La innovación digital es el motor del cambio. Las herramientas modernas permiten 
automatizar procesos complejos. El software actual es más intuitivo. 
Sin embargo, la brecha de acceso sigue siendo un problema social.
"""

doc_reto = nlp(texto_analizar)
objetivo = nlp(concepto_clave)

analisis_semantico = []
for token in doc_reto:
    if not token.is_stop and not token.is_punct and len(token.text) > 2:
        analisis_semantico.append({
            "Palabra": token.text,
            "Afinidad con 'tecnología'": objetivo.similarity(token)
        })

df_reto = pd.DataFrame(analisis_semantico).sort_values(by="Afinidad con 'tecnología'", ascending=False)
print("Mapa de afinidad semántica:")
print(df_reto.head(10))

---
## Conclusiones

- La terminología profesional se extrae mediante **patrones gramaticales**, no solo frecuencias de palabras sueltas.
- Los **vectores** permiten que la máquina comprenda el parentesco entre palabras por su contexto de uso.
- Como profesionales de la lengua, estas herramientas nos permiten pasar de la búsqueda de texto a la **búsqueda de conceptos**.

**Siguiente paso:** Estos vectores y patrones son los que permiten que modelos más grandes (LLMs) puedan generar texto con coherencia. Lo exploraremos en el bloque de **IA Generativa**.